In [ ]:
import eval_common as _ec
_ec.set_axis("mechanism")     # change to "lob" for the line-of-business axis IGNORE
AXIS = _ec.AXIS
print("AXIS =", AXIS, "| categories:", _ec.MECH_ORDER)

AXIS = mechanism | categories: ['Reliability', 'Bias & Fairness', 'Privacy, Confidentiality & Infringement', 'Security & Misuse', 'Autonomous Actions', 'Governance, Oversight & Explainability']


In [7]:
from pathlib import Path
import numpy as np, pandas as pd
OUTPUT_DIR=Path("model_outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
DATE_COL="date"
import eval_common as _ec
MECH_ORDER = _ec.MECH_ORDER
df = _ec.load_incidents()
counts = _ec.build_counts(df)
print(counts.shape, counts.index[0], "->", counts.index[-1])

(127, 6) 2016-01 -> 2026-07


In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
evaluate = _ec.evaluate
mlog = _ec.mlog

In [ ]:
import statsmodels.api as sm

def panelize(C):
    rows = []
    t = np.arange(len(C), dtype=float)
    N = C.sum(axis=1)
    for category in MECH_ORDER:
        rows.append(pd.DataFrame({
            "month_period": C.index,
            "month_index": t,
            "year": C.index.year,
            "category": category,
            "count": C[category].to_numpy(dtype=float),
            "N": N.to_numpy(dtype=float),
        }))
    panel = pd.concat(rows, ignore_index=True)
    panel["share"] = panel["count"] / panel["N"].replace(0, np.nan)
    return panel

panel = panelize(counts)

def X(t, degree):
    z = np.asarray(t, dtype=float) / 120.0
    design = pd.DataFrame({"const": np.ones(len(z), dtype=float), "time": z})
    if degree == 2:
        design["time_squared"] = z ** 2
    return design.reset_index(drop=True)

def fit(g, degree):
    g = g.sort_values("month_index").reset_index(drop=True).copy()
    y = g["count"].to_numpy(dtype=float)
    exposure = g["N"].clip(lower=1).to_numpy(dtype=float)
    if len(g) == 0 or y.sum() <= 0 or np.count_nonzero(y) < 3:
        return None
    design = X(g["month_index"].to_numpy(dtype=float), degree)
    offset = np.log(exposure)
    try:
        result = sm.GLM(endog=y, exog=design, family=sm.families.Poisson(),
                        offset=offset).fit(cov_type="HC0", maxiter=200)
        if not np.isfinite(result.aic):
            return None
        if not np.all(np.isfinite(np.asarray(result.params, dtype=float))):
            return None
        return result
    except (ValueError, np.linalg.LinAlgError, FloatingPointError):
        return None

def choose(g):
    candidates = []
    for degree in (1, 2):
        result = fit(g, degree)
        if result is not None:
            candidates.append((degree, result))
    if not candidates:
        return None, None, "historical_share_fallback"
    degree, result = min(candidates, key=lambda item: item[1].aic)
    return degree, result, "poisson"

def historical_shares(training_panel, smoothing=0.5):
    totals = (training_panel.groupby("category")["count"].sum()
              .reindex(MECH_ORDER, fill_value=0.0).astype(float))
    totals = totals + smoothing
    return totals / totals.sum()

rows = []
specs = []
for (yr, half) in _ec.TEST_FOLDS:
    test_start, test_end = _ec.fold_bounds(yr, half)
    fold = _ec.fold_label(yr, half)
    tr = panel[panel["month_period"] < test_start].copy()
    te = panel[(panel["month_period"] >= test_start) & (panel["month_period"] <= test_end)].copy()
    if tr.empty or te.empty:
        print(f"Skipping {fold}: missing training or test data.")
        continue

    fallback = historical_shares(tr, smoothing=0.5)
    models = {}
    for category in MECH_ORDER:
        category_train = tr[tr["category"] == category].copy()
        degree, result, status = choose(category_train)
        models[category] = {"degree": degree, "result": result, "status": status}
        specs.append({
            "fold": fold, "category": category, "selected_degree": degree,
            "aic": result.aic if result is not None else np.nan,
            "status": status,
            "total_training_count": float(category_train["count"].sum()),
            "nonzero_training_months": int(category_train["count"].gt(0).sum()),
        })

    for month_period, g in te.groupby("month_period"):
        g = g.set_index("category").reindex(MECH_ORDER)
        month_total = float(g["N"].dropna().iloc[0])
        if month_total <= 0:
            continue
        mus = {}
        for category in MECH_ORDER:
            model_info = models[category]
            if model_info["status"] == "poisson":
                row = g.loc[category]
                design_test = X([row["month_index"]], model_info["degree"])
                predicted_mean = float(model_info["result"].predict(
                    design_test, offset=np.log([max(month_total, 1.0)]))[0])
                mus[category] = max(predicted_mean, 1e-12)
            else:
                mus[category] = float(fallback[category]) * month_total
        denominator = sum(mus.values())
        if denominator <= 0:
            continue
        actual_counts = g["count"].fillna(0.0).astype(float)
        actual_total = actual_counts.sum()
        if actual_total <= 0:
            continue
        actual_shares = actual_counts / actual_total
        for category in MECH_ORDER:
            rows.append({
                "model": "tuned_offset_poisson",
                "fold": fold,
                "month_period": str(month_period),
                "category": category,
                "actual_count": float(actual_counts[category]),
                "actual_share": float(actual_shares[category]),
                "predicted_share": float(mus[category] / denominator),
                "fitting_status": models[category]["status"],
                "selected_degree": models[category]["degree"],
            })

pred = pd.DataFrame(rows)
specs = pd.DataFrame(specs)
overall, by_fold = evaluate(pred)
display(overall)
display(by_fold)
display(specs[specs["status"] != "poisson"])

pred.to_csv(OUTPUT_DIR / f"predictions_poisson_{AXIS}.csv", index=False)
specs.to_csv(OUTPUT_DIR / f"poisson_selected_specs_{AXIS}.csv", index=False)
overall.to_csv(OUTPUT_DIR / f"metrics_poisson_overall_{AXIS}.csv", index=False)
by_fold.to_csv(OUTPUT_DIR / f"metrics_poisson_byfold_{AXIS}.csv", index=False)

,model,mae,rmse,mean_log_score_per_incident
0,tuned_offset_poisson,0.096433,0.131429,-1.672613


,model,fold,mae,rmse
0,tuned_offset_poisson,2020-H1,0.152409,0.203595
1,tuned_offset_poisson,2020-H2,0.130078,0.162158
2,tuned_offset_poisson,2021-H1,0.122954,0.148332
3,tuned_offset_poisson,2021-H2,0.126480,0.156049
4,tuned_offset_poisson,2022-H1,0.097533,0.121739
5,tuned_offset_poisson,2022-H2,0.100471,0.127562
6,tuned_offset_poisson,2023-H1,0.105994,0.149319
7,tuned_offset_poisson,2023-H2,0.091540,0.117254
8,tuned_offset_poisson,2024-H1,0.075580,0.101290
9,tuned_offset_poisson,2024-H2,0.056185,0.073248


Specifications and fallback checks:


,fold,category,selected_degree,aic,status,total_training_count,nonzero_training_months
0,2020-H1,Reliability,1,72.529873,poisson,18.0,15
1,2020-H1,Bias & Fairness,1,117.463488,poisson,44.0,26
2,2020-H1,"Privacy, Confidentiality & Infringement",1,113.088560,poisson,45.0,29
3,2020-H1,Security & Misuse,1,70.829472,poisson,17.0,13
4,2020-H1,Autonomous Actions,1,124.092740,poisson,47.0,29
...,...,...,...,...,...,...,...
73,2026-H1,Bias & Fairness,2,337.931693,poisson,161.0,85
74,2026-H1,"Privacy, Confidentiality & Infringement",2,340.895751,poisson,194.0,89
75,2026-H1,Security & Misuse,1,363.811572,poisson,585.0,80
76,2026-H1,Autonomous Actions,2,347.744269,poisson,183.0,91


Categories using fallback:


,fold,category,selected_degree,aic,status,total_training_count,nonzero_training_months
